# Pipeline A - Mid-fusion (recommended)

`BEV_camera` (stereo-splat) + `BEV_lidar` (PointPillars) -> concat + conv -> CenterPoint head.

Each section just **calls the project `.py` modules** - no heavy code lives in the
notebook. Stages we haven't built yet (`globals.py`, the fusion + head in
`network.py`, `train.py`, `evaluation.py`) are left empty as TODO.

## 1. Imports

In [ ]:
import numpy as np
import torch

# Project modules — the implementation stays inside the .py files (6-file layout:
# data / evaluation / globals / network / train / utils).
import data, utils
import globals as G                       # shared BEV grid + channel contract (single source)
from data import Py123dDataset
from network import PillarConfig, PointPillarsBranch, StereoBEVBranch   # full network

# Stages still to implement (empty 0-byte modules) — imported as placeholders.
import train        # noqa: F401  (empty: training loop TODO)
import evaluation   # noqa: F401  (center-distance AP eval, see evaluation.py)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

## 2. Globals

In [ ]:
# globals.py is the single source of truth for the shared BEV grid + channel
# contract. Every branch config (PillarConfig, MonoBEVConfig, StereoBEVConfig)
# defaults to it, so the grid can't drift between branches. Indexing: ix*ny+iy.
print("grid_size (nx, ny):", G.GRID_SIZE, "| x_range:", G.X_RANGE, "| y_range:", G.Y_RANGE)
print("channels — camera:", G.CAMERA_BEV_CHANNELS, "| lidar:", G.LIDAR_BEV_CHANNELS,
      "| num_classes:", G.NUM_CLASSES)

## 3. Utils

In [ ]:
# Visualisation helpers live in utils.py (LiDAR density BEV + GT boxes, point
# cloud, stereo pair, frustum, voxels, clusters).
import utils
print([f for f in dir(utils) if f.startswith("visualize_")])
# e.g. utils.visualize_bev(sample)  -> ego-frame boxes over a LiDAR BEV

## 4. Data

In [ ]:
# data.py loads KITTI-360 through py123d and assembles one StereoSample / frame.
dataset = Py123dDataset(split_names=["kitti360_train"], max_num_scenes=1)
frame   = dataset.get_frame(0, dataset.scenes[0].number_of_history_iterations + 13)
sample  = frame.to_stereo_sample()
print(sample)

# GT boxes: global frame (for 2D projection) + ego frame (already converted, for
# BEV target encoding). See data.boxes_global_to_ego / assert_boxes_in_sensor_range.
print("boxes_3d (global):", sample.boxes_3d.shape,
      "| boxes_3d_ego:", sample.boxes_3d_ego.shape)
# utils.visualize_images(sample); utils.visualize_pointcloud(sample)

## 5. Network

In [ ]:
# --- Stage A: the two BEV branches emit grid-aligned (C, nx, ny) maps ---------
lidar_branch  = PointPillarsBranch(PillarConfig()).to(DEVICE).eval()
camera_branch = StereoBEVBranch().to(DEVICE).eval()

# PointPillars wants (N, 4) = [x, y, z, intensity], so append the reflectance.
pts_lidar = np.concatenate(
    [sample.lidar_xyz, sample.lidar_features["intensity"][:, None]], axis=1
).astype(np.float32)
with torch.no_grad():
    bev_lidar  = lidar_branch(pts_lidar, device=DEVICE)         # (C_lidar, nx, ny)
    bev_camera = camera_branch(sample, device=DEVICE)            # (C_cam,   nx, ny)
print("BEV_lidar:", tuple(bev_lidar.shape), "| BEV_camera:", tuple(bev_camera.shape))

# --- BEV fusion + 2D CenterPoint head (network.py) ----------------------------
# Build the detector straight from the Stage A outputs, then print the contract:
# this is the "see the parameters, then go back to Stage A" loop.
from network import BEVDetector, describe

detector = BEVDetector.from_bev_maps(bev_camera, bev_lidar, num_classes=3).to(DEVICE).eval()
describe(detector)          # prints what Stage A must emit + parameter counts

with torch.no_grad():
    out = detector(bev_camera, bev_lidar)   # concat + conv -> head
print({k: tuple(v.shape) for k, v in out.items()})
#   heatmap (1, num_classes, nx, ny)  +  offset (1, 2, nx, ny)

# Swap to Pipeline C later: BEVDetector.from_bev_maps(..., fusion_cls=CrossAttentionFusion)
# TODO (target encoder): rasterize sample.boxes_3d_ego[:, :2] + class into the same
#   (num_classes, nx, ny) heatmap + (2, nx, ny) offset targets to train against.

## 6. Train

In [ ]:
# TODO: train.py is empty.
# Warm-start the two single-sensor BEV baselines, then fine-tune the fused model
# end-to-end. CenterPoint loss: focal heatmap + L1 (x, y) offset + CE class.

## 7. Test

In [ ]:
# TODO: evaluation.py is empty.
# Center-distance AP @0.5/1/2/4 m + CDS, per-range bins, per-class,
# plus Orin latency. Reuse the same harness for the single-sensor baselines.